# GPT-2 from Scratch, Initialized with Pretrained GPT-2 Weights

This builds a **decoder-only, GPT-2-style Transformer** entirely from scratch (no
`transformers.GPT2Model` used internally) and then loads the official pretrained
GPT-2 weights (from Hugging Face, OpenAI's checkpoint) into it, weight-by-weight.

Architecture differences from the encoder-decoder `Transformer` in the other
notebook:
- **Decoder-only**: no encoder, no cross-attention — just causal self-attention.
- **Pre-LayerNorm**: `x = x + attn(ln_1(x))`, `x = x + mlp(ln_2(x))` (GPT-2 uses
  pre-norm, the earlier notebook used post-norm).
- **GELU** activation (tanh approximation, same as OpenAI's original) instead of ReLU.
- **Learned absolute position embeddings** (`wpe`), added to token embeddings (`wte`).
- **Weight tying**: the output projection (`lm_head`) shares weights with `wte`.
- **Fused QKV projection**: one `Linear(n_embd, 3*n_embd)` instead of three separate
  `Linear` layers, split afterward — this is how GPT-2 actually stores it.
- GPT-2's original weights were saved from TensorFlow `Conv1D` layers, whose weight
  matrices are stored **transposed** relative to PyTorch's `nn.Linear`. We transpose
  the four affected weight matrices per block when loading.


In [ ]:
!pip install transformers -q


In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 1. Config

Matches the shapes of the four public GPT-2 checkpoints. `block_size` is GPT-2's
context length (1024), `vocab_size` is 50257 (50000 BPE merges + 256 byte tokens +
1 `<|endoftext|>` token).


In [ ]:
@dataclass
class GPT2Config:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0   # 0.0 for inference / fine-tuning with pretrained weights
    bias: bool = True      # GPT-2 uses biases on all Linears and LayerNorms


# n_layer, n_head, n_embd for each released size
GPT2_SIZES = {
    "gpt2":        dict(n_layer=12, n_head=12, n_embd=768),   # 124M params
    "gpt2-medium": dict(n_layer=24, n_head=16, n_embd=1024),  # 350M params
    "gpt2-large":  dict(n_layer=36, n_head=20, n_embd=1280),  # 774M params
    "gpt2-xl":     dict(n_layer=48, n_head=25, n_embd=1600),  # 1558M params
}


## 2. Causal self-attention

One fused `c_attn` projection produces Q, K, V together (`n_embd -> 3*n_embd`), split
along the last dim, reshaped into heads. A causal mask (registered as a buffer, not a
parameter) ensures position `t` can only attend to positions `<= t`.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        # fused q, k, v projection -- matches GPT-2's checkpoint layout exactly
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # causal mask, (1, 1, block_size, block_size); not a learned parameter
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer("bias_mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x, use_cache=False, past_kv=None):
        B, T, C = x.size()

        qkv = self.c_attn(x)                       # (B, T, 3*C)
        q, k, v = qkv.split(self.n_embd, dim=2)     # each (B, T, C)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # (B, nh, T, T_kv)
        Tq, Tk = att.size(-2), att.size(-1)
        # causal mask aligned to the right (works whether or not we have cached past kv)
        mask = self.bias_mask[:, :, Tk - Tq:Tk, :Tk]
        att = att.masked_fill(mask == 0, torch.finfo(att.dtype).min)
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        out = att @ v                                              # (B, nh, T, hd)
        out = out.transpose(1, 2).contiguous().view(B, T, C)        # (B, T, C)
        out = self.resid_dropout(self.c_proj(out))
        return out, present


## 3. MLP block

GPT-2's feed-forward network: `n_embd -> 4*n_embd -> n_embd` with GELU (tanh
approximation — the same one OpenAI's original TensorFlow code used).


In [ ]:
class MLP(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


## 4. Transformer block (pre-LayerNorm)

```
x = x + attn(ln_1(x))
x = x + mlp(ln_2(x))
```

Note the residual stream itself is never normalized — only the branch inputs are.
This is the key structural difference from the post-norm encoder/decoder layers in
the first notebook.


In [ ]:
class Block(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x, use_cache=False, past_kv=None):
        attn_out, present = self.attn(self.ln_1(x), use_cache=use_cache, past_kv=past_kv)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, present


## 5. Full GPT-2 model

Token embedding (`wte`) + learned position embedding (`wpe`) -> dropout -> stack of
`Block`s -> final LayerNorm (`ln_f`) -> `lm_head`. `lm_head.weight` is **tied** to
`wte.weight` (standard GPT-2 practice: it roughly halves the embedding-related
parameter count and tends to improve quality).


In [ ]:
class GPT2(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd, bias=config.bias)

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.apply(self._init_weights)
        # GPT-2 paper: scale residual projection weights by 1/sqrt(2*n_layer)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, past_kvs=None):
        B, T = idx.size()
        assert T <= self.config.block_size, (
            f"sequence length {T} exceeds block_size {self.config.block_size}"
        )

        past_len = past_kvs[0][0].size(2) if past_kvs is not None else 0
        pos = torch.arange(past_len, past_len + T, device=idx.device)

        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        x = self.drop(x)

        presents = [] if use_cache else None
        for i, block in enumerate(self.h):
            past_kv = past_kvs[i] if past_kvs is not None else None
            x, present = block(x, use_cache=use_cache, past_kv=past_kv)
            if use_cache:
                presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss, presents

    def num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.wpe.weight.numel()
        return n


## 6. Loading pretrained GPT-2 weights

We pull the official checkpoint via `transformers.GPT2LMHeadModel.from_pretrained`,
then copy each tensor into our model by name. The only subtlety: Hugging Face keeps
GPT-2's original TF-style `Conv1D` layers for `c_attn`, `c_proj` (in attention) and
`c_fc`, `c_proj` (in the MLP). `Conv1D.weight` has shape `(in_features, out_features)`
— the **transpose** of `nn.Linear.weight`'s `(out_features, in_features)` — so those
four weight matrices need `.t()` before copying.


In [ ]:
from transformers import GPT2LMHeadModel


@torch.no_grad()
def gpt2_from_pretrained(model_type: str = "gpt2"):
    assert model_type in GPT2_SIZES, f"unknown model_type {model_type!r}"
    print(f"loading weights from pretrained GPT-2: {model_type}")

    config = GPT2Config(vocab_size=50257, block_size=1024, **GPT2_SIZES[model_type])
    model = GPT2(config)
    sd = model.state_dict()
    sd_keys = [k for k in sd.keys() if not k.endswith(".bias_mask")]

    hf_model = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = hf_model.state_dict()
    sd_keys_hf = [k for k in sd_hf.keys() if not k.endswith(".attn.bias") and not k.endswith(".attn.masked_bias")]

    # huggingface GPT2's state dict uses `transformer.` prefix and slightly
    # different block naming (`transformer.h.{i}....`) plus `lm_head.weight`
    # at the top level -- but since lm_head is tied to wte in both models,
    # we only need to copy the `transformer.*` keys.
    sd_keys_hf = [k for k in sd_keys_hf if k.startswith("transformer.")]

    assert len(sd_keys_hf) == len(sd_keys), (
        f"mismatched key counts: {len(sd_keys_hf)} (hf) vs {len(sd_keys)} (ours)"
    )

    # these four weight types are stored transposed in HF's checkpoint (Conv1D)
    transposed = ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight")

    for k_ours in sd_keys:
        k_hf = "transformer." + k_ours
        src = sd_hf[k_hf]
        dst = sd[k_ours]
        if any(k_ours.endswith(w) for w in transposed):
            assert src.shape[::-1] == dst.shape, f"shape mismatch at {k_ours}: {src.shape} vs {dst.shape}"
            dst.copy_(src.t())
        else:
            assert src.shape == dst.shape, f"shape mismatch at {k_ours}: {src.shape} vs {dst.shape}"
            dst.copy_(src)

    print(f"loaded {len(sd_keys)} tensors, {model.num_params()/1e6:.1f}M non-embedding params")
    return model, config


In [ ]:
model, gpt2_config = gpt2_from_pretrained("gpt2")
model.to(device)
model.eval()


## 7. Sanity check against Hugging Face's own forward pass

If the weight loading is correct, our from-scratch model's logits should match
`GPT2LMHeadModel`'s logits almost exactly (up to floating-point noise).


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
hf_ref_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()

prompt = "The quick brown fox"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    our_logits, _, _ = model(ids)
    hf_logits = hf_ref_model(ids).logits

max_abs_diff = (our_logits - hf_logits).abs().max().item()
print(f"max |logit diff| between our model and HF's: {max_abs_diff:.6f}")
assert max_abs_diff < 1e-3, "weight loading looks wrong -- logits should match closely"
print("Match confirmed: our from-scratch GPT-2 reproduces the reference model's outputs.")


## 8. Text generation (autoregressive sampling with KV cache)

Standard sampling loop: temperature scaling, optional top-k filtering, multinomial
sampling. Uses the `past_kvs`/`use_cache` plumbing built into `CausalSelfAttention`
so each new token only recomputes attention for the new position, not the whole
sequence.


In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8, top_k=50):
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    past_kvs = None
    cur_ids = ids
    for _ in range(max_new_tokens):
        # after the first step, we only need to feed the newest token (cache holds the rest)
        step_input = cur_ids if past_kvs is None else cur_ids[:, -1:]
        logits, _, past_kvs = model(step_input, use_cache=True, past_kvs=past_kvs)

        next_logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(next_logits, top_k)
            next_logits[next_logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        cur_ids = torch.cat([cur_ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(cur_ids[0].tolist())


torch.manual_seed(42)
print(generate(model, tokenizer, "The quick brown fox", max_new_tokens=40))


## 9. (Optional) Load a bigger checkpoint

Swap in `"gpt2-medium"`, `"gpt2-large"`, or `"gpt2-xl"` — the same `gpt2_from_pretrained`
function handles all four sizes since `GPT2_SIZES` captures the per-size shape config.

```python
model_lg, config_lg = gpt2_from_pretrained("gpt2-medium")
model_lg.to(device).eval()
print(generate(model_lg, tokenizer, "Once upon a time", max_new_tokens=40))
```
